# Context Engineering 與 AI Agent 基礎實作

**課程**: Session 1 - Technical Foundations  
**目標**: 理解 Context Engineering 原理，實作簡易 AI Agent

---

## Lab 結構

- **Exercise 1**: 基礎對話 Agent - 理解 Context 累積
- **Exercise 2**: System Prompt 實驗 - 結構化的重要性

---

## 環境設定

### 🎯 準備工作

- 安裝必要套件
- 設定 API Key
- 載入工具函式

---

In [ ]:
# 安裝必要的套件
!pip install google-genai -q

print("✓ 套件安裝完成！")

In [ ]:
# 匯入必要的函式庫
from google import genai
from google.genai import types

# 設定 API Key
# 請在 Colab 的 Secrets 中設定 GOOGLE_API_KEY
from google.colab import userdata
api_key = userdata.get('GOOGLE_API_KEY')

# 初始化 Gemini client
client = genai.Client(api_key=api_key)

# 使用的模型
MODEL_NAME = "gemini-2.0-flash"

print("✓ API 設定完成！")
print(f"✓ 使用模型: {MODEL_NAME}")

---

## Exercise 1: 基礎對話 Agent

### 🎯 學習目標

- 理解 Context 如何累積
- 觀察 Token 使用量隨對話增長
- 體驗「手動管理記憶」的概念

---

### Task 1.1: 實作基礎對話循環

**重點**: 
- 每次請求都要帶上完整的對話歷史
- 追蹤 Token 使用量

In [ ]:
class SimpleAgent:
    """
    簡易對話 Agent (使用 Gemini API)
    """
    def __init__(self, system_prompt="你是一個有幫助的 AI 助手。"):
        self.system_prompt = system_prompt
        self.conversation_history = []
        self.token_usage_history = []
    
    def chat(self, user_message):
        """
        發送訊息並取得回應
        """
        # 將用戶訊息加入歷史
        self.conversation_history.append({
            "role": "user",
            "parts": [{"text": user_message}]
        })
        
        # 呼叫 Gemini API
        response = client.models.generate_content(
            model=MODEL_NAME,
            contents=self.conversation_history,
            config=types.GenerateContentConfig(
                system_instruction=self.system_prompt,
                max_output_tokens=1024,
            )
        )
        
        # 取得回應
        assistant_message = response.text
        
        # 將助手回應加入歷史
        self.conversation_history.append({
            "role": "model",
            "parts": [{"text": assistant_message}]
        })
        
        # 記錄 Token 使用量
        usage = response.usage_metadata
        self.token_usage_history.append({
            "turn": len(self.token_usage_history) + 1,
            "input_tokens": usage.prompt_token_count,
            "output_tokens": usage.candidates_token_count,
            "total_tokens": usage.total_token_count
        })
        
        return assistant_message
    
    def show_token_usage(self):
        """
        顯示 Token 使用統計
        """
        print("\n" + "=" * 80)
        print("📊 Token 使用統計")
        print("=" * 80)
        print(f"{'輪次':<8} {'輸入 Token':<15} {'輸出 Token':<15} {'總計':<15}")
        print("-" * 80)
        
        for usage in self.token_usage_history:
            print(f"{usage['turn']:<8} {usage['input_tokens']:<15} {usage['output_tokens']:<15} {usage['total_tokens']:<15}")
        
        # 總計
        total_input = sum(u['input_tokens'] for u in self.token_usage_history)
        total_output = sum(u['output_tokens'] for u in self.token_usage_history)
        total = total_input + total_output
        
        print("-" * 80)
        print(f"{'總計':<8} {total_input:<15} {total_output:<15} {total:<15}")
        print("=" * 80)
        
        # 成長趨勢分析
        if len(self.token_usage_history) > 1:
            first_input = self.token_usage_history[0]['input_tokens']
            last_input = self.token_usage_history[-1]['input_tokens']
            
            print(f"\n📈 Context 成長趨勢:")
            print(f"   第 1 輪輸入: {first_input} tokens")
            print(f"   第 {len(self.token_usage_history)} 輪輸入: {last_input} tokens")
            
            # 🔥 防止除以零錯誤
            if first_input > 0:
                growth_rate = ((last_input - first_input) / first_input) * 100
                print(f"   成長率: {growth_rate:.1f}%")
            else:
                print(f"   成長: {last_input - first_input} tokens")
    
    def reset(self):
        """
        清空對話歷史
        """
        self.conversation_history = []
        self.token_usage_history = []

print("✓ SimpleAgent 類別定義完成")

### Task 1.2: 實際使用 SimpleAgent

**重點**: 
- 進行多輪對話，觀察 Context 如何累積
- 檢視 Token 使用量隨對話增長的變化

In [ ]:
# 建立一個 SimpleAgent 實例
agent = SimpleAgent(system_prompt="你是一個友善的 AI 助手，請用繁體中文回答問題。")

# 進行多輪對話
conversations = [
    "你好！我叫 Alice，是一名軟體工程師。",
    "我最近在學習 Python，你有什麼建議嗎？",
    "對了，你還記得我的名字嗎？",
    "我的職業是什麼？"
]

for user_msg in conversations:
    print(f"\n{'=' * 60}")
    print(f"User: {user_msg}")
    response = agent.chat(user_msg)
    print(f"Assistant: {response}")

# 顯示 Token 使用統計
agent.show_token_usage()

### Task 1.3: 驗證「記憶」來自 Context

**重點**: 
- 清空 Context 後，AI 會「失憶」
- 證明 LLM 本身沒有記憶，「記憶」來自 Context

In [ ]:
# 清空 Context，重新提問
print("=" * 60)
print("清空 Context...")
print("=" * 60)

agent.reset()

# 再次詢問相同問題
print("\nUser: 你還記得我的名字嗎？")
response = agent.chat("你還記得我的名字嗎？")
print(f"Assistant: {response}")

print("\n" + "=" * 60)
print("觀察: AI 完全「失憶」了！")
print("這證明 LLM 的記憶 = Context，而不是模型本身記住的")
print("=" * 60)

### ✅ Exercise 1 完成檢查點

**你學到了**:
- ✓ LLM 的「記憶」是透過 Context 實現的
- ✓ Token 使用量隨對話增長而增加
- ✓ 清空 Context = AI 失憶

---

## Exercise 2: System Prompt 實驗

### 🎯 學習目標

- 理解 System Prompt 的重要性
- 對比結構化 vs 非結構化 Prompt
- 觀察輸出一致性的差異

---

### Task 2.1: 無結構 vs 有結構 System Prompt

In [ ]:
# 無結構的 System Prompt
unstructured_prompt = "你是程式碼審查助手，幫助審查程式碼。"

# 有結構的 System Prompt
structured_prompt = """<role>你是專業的程式碼審查助手</role>

<rules>
  - 關注安全性與效能問題
  - 提供具體改進建議
  - 使用 Markdown 格式輸出
</rules>

<output_format>
## 問題
[列出發現的問題]

## 建議
[提供改進建議]

## 範例
[如果適用，提供程式碼範例]
</output_format>"""

print("✓ System Prompt 準備完成")

In [ ]:
# 測試程式碼
test_code = """
def login(username, password):
    query = f"SELECT * FROM users WHERE username='{username}' AND password='{password}'"
    result = db.execute(query)
    return result
"""

print("測試程式碼:")
print(test_code)

In [ ]:
# 測試 1: 無結構 System Prompt
print("=" * 80)
print("測試 1: 無結構 System Prompt")
print("=" * 80)

agent_unstructured = SimpleAgent(system_prompt=unstructured_prompt)
response1 = agent_unstructured.chat(f"請審查這段程式碼:\n{test_code}")

print(f"\n🤖 回應:\n{response1}")
print("\n" + "=" * 80)

In [ ]:
# 測試 2: 結構化 System Prompt
print("=" * 80)
print("測試 2: 結構化 System Prompt")
print("=" * 80)

agent_structured = SimpleAgent(system_prompt=structured_prompt)
response2 = agent_structured.chat(f"請審查這段程式碼:\n{test_code}")

print(f"\n🤖 回應:\n{response2}")
print("\n" + "=" * 80)

### 📊 觀察結果

**結構化 System Prompt 的優勢**:

1. **輸出格式一致**: 使用指定的 Markdown 結構
2. **內容更聚焦**: 按照 rules 關注安全性和效能
3. **易於解析**: 程式可以更容易處理結構化輸出

**💡 結構化 = 更可控、更穩定的 AI 行為**

---

### ✅ Exercise 2 完成檢查點

**你學到了**:
- ✓ 結構化 System Prompt 提升輸出一致性
- ✓ XML 標籤可以清晰劃分不同部分
- ✓ 明確的輸出格式讓 AI 更容易遵循

---